In [1]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from roboflow import Roboflow

rf = Roboflow(api_key="64g4PSrg0kTCRLoALd8j")

# Seat Belt and Mobile Object Detection dataset
project = rf.workspace("aiactive20092009-gmail-com").project("seat_belt-and-mobile")
dataset = project.version("1").download("yolov11")

DATASET_DIR = dataset.location
print("Dataset downloaded to:", DATASET_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to seat_belt-and-mobile-1 in yolov11:: 100%|██████████| 2241/2241 [00:00<00:00, 6437.13it/s]

Dataset downloaded to: /content/seat_belt-and-mobile-1


In [7]:
import torch

MODEL_ARCH        = 'yolo11n.pt'          # pretrained, NOT .yaml
EPOCHS            = 100
BATCH_SIZE        = 16
IMG_SIZE          = 640
CONFIDENCE_THRESHOLD = 0.25
PROJECT_NAME      = '/content/drive/MyDrive/YOLO11-Seatbelt'  # saves to Drive
EXPERIMENT_NAME   = 'exp1'
DATA_CONFIG       = f'{DATASET_DIR}/data.yaml'  # Roboflow generates this automatically

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Data config: {DATA_CONFIG}")

Using device: cuda
Data config: /content/seat_belt-and-mobile-1/data.yaml


In [9]:
from ultralytics import YOLO

model = YOLO(MODEL_ARCH)

train_results = model.train(
    data=DATA_CONFIG,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    device=device,
    exist_ok=True,
    patience=15,
    optimizer='SGD',
    lr0=0.001,
    lrf=0.01,
)

print("\nTraining completed!\n")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/seat_belt-and-mobile-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patie

In [10]:
best_weights_path = f'{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt'
model = YOLO(best_weights_path)

validation_results = model.val(
    data=DATA_CONFIG,
    split='val',
    conf=CONFIDENCE_THRESHOLD,
)

metrics = validation_results.box
precision = getattr(metrics, 'mp', None)
recall    = getattr(metrics, 'mr', None)
map50     = getattr(metrics, 'map50', None)

f1 = 2 * (precision * recall) / (precision + recall) if precision and recall else None

print(f"Precision : {precision:.2f}")
print(f"Recall    : {recall:.2f}")
print(f"mAP50     : {map50:.2f}")
print(f"F1 Score  : {f1:.2f}")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2284.0±1025.7 MB/s, size: 65.1 KB)
val: Scanning /content/seat_belt-and-mobile-1/valid/labels.cache... 337 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 337/337 141.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 4.3it/s 5.1s
                   all        337        675      0.941      0.856      0.871      0.521
                mobile        131        131      0.895       0.71      0.743      0.287
              seatbelt        204        204      0.927      0.867      0.874      0.422
            windshield        337        340          1      0.991      0.995      0.856
Speed: 2.2ms preprocess, 5.5ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/runs/detec